In [8]:
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import statsmodels.api as sm

from Regression_utils import *
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


In [9]:
main_df = pd.read_excel(r"C:\Users\jroland\East Texas Salt Water Disposal Company\Operations - Documents\Engineering\Disposal Wells\Workover Analysis\Derived All Data.xlsx")
print(main_df.columns)

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Name', 'Year-Month', 'Hours',
       'Water Total', 'PSI', 'Revenue', 'Metered DM', 'Power Factor',
       'Billed DM', 'Billing KWH', 'Fuel Rate', 'Electrical Bill',
       'Workover Info', 'Workover Cost', 'Pump Model', 'Pump Number',
       'Plunger Size', 'Install Date', 'Maintenance Cost',
       'Maintenance Comments', 'KWH per Barrel', 'Cost per KWH',
       'Electrical Cost per Barrel', 'Maintenance Cost per Barrel',
       'Workover Cost per Barrel', 'Revenue per Barrel', 'Income per Barrel',
       'Income', 'Total Cost', '6 Month Avg Water Total', '6 Month Avg KWH',
       '6 Month Avg Revenue', '6 Month Avg Income', '6 Month Avg Total Cost',
       '6 Month Avg KWH per Barrel Weighted', '6 Month Avg Cost per KWH',
       '6 Month Avg Electrical Cost per Barrel',
       '6 Month Avg Maintenance Cost per Barrel',
       '6 Month Avg Workover Cost per Barrel',
       '6 Month Avg Revenue per Barrel', '6 Month Avg Income per Barrel',
      

In [10]:
# Double check that there are no problematic values for our data
print(main_df[main_df["PSI"].isna()])
print(np.isinf(main_df["PSI"]).sum())
print(main_df[main_df["Billing KWH"].isna()])
print(main_df["Billing KWH"].isna().sum())
print(np.isinf(main_df["Billing KWH"]).sum())

Empty DataFrame
Columns: [Unnamed: 0.1, Unnamed: 0, Name, Year-Month, Hours, Water Total, PSI, Revenue, Metered DM, Power Factor, Billed DM, Billing KWH, Fuel Rate, Electrical Bill, Workover Info, Workover Cost, Pump Model, Pump Number, Plunger Size, Install Date, Maintenance Cost, Maintenance Comments, KWH per Barrel, Cost per KWH, Electrical Cost per Barrel, Maintenance Cost per Barrel, Workover Cost per Barrel, Revenue per Barrel, Income per Barrel, Income, Total Cost, 6 Month Avg Water Total, 6 Month Avg KWH, 6 Month Avg Revenue, 6 Month Avg Income, 6 Month Avg Total Cost, 6 Month Avg KWH per Barrel Weighted, 6 Month Avg Cost per KWH, 6 Month Avg Electrical Cost per Barrel, 6 Month Avg Maintenance Cost per Barrel, 6 Month Avg Workover Cost per Barrel, 6 Month Avg Revenue per Barrel, 6 Month Avg Income per Barrel, Cumulative Electrical Cost, Cumulative Workover Cost, Cumulative Maintenance Cost, Cumulative Total Cost, Cumulative Income, Cumulative Revenue]
Index: []

[0 rows x 49 co

In [19]:
# Standard Linear Regression
# EXPLANATORY
# add intercept for regression
exog = main_df["PSI"]
exog = sm.add_constant(exog)
endog = main_df["Billing KWH"]

# kwh vs psi
print(exog, endog)
lm = sm.OLS(endog, exog).fit()
print("Billing KWH explained by PSI", lm.summary())

# electrical cost vs psi
endog = main_df["Electrical Bill"]
lm = sm.OLS(endog, exog).fit()
print("Electrical bill costs explained by PSI", lm.summary())

# kwh vs water total
endog = main_df["Billing KWH"]
exog = main_df["Water Total"]
exog = sm.add_constant(exog)
lm = sm.OLS(endog, exog).fit()
print("KWH explained by water volume", lm.summary())

      const         PSI
0       1.0  772.857143
1       1.0  777.142857
2       1.0  782.857143
3       1.0  774.285714
4       1.0  773.333333
...     ...         ...
8521    1.0  754.440000
8522    1.0  772.220000
8523    1.0  810.000000
8524    1.0    0.000000
8525    1.0    0.000000

[8526 rows x 2 columns] 0       125200.0
1       129440.0
2       148320.0
3       129440.0
4       127120.0
          ...   
8521     53040.0
8522     56400.0
8523     40560.0
8524        80.0
8525        40.0
Name: Billing KWH, Length: 8526, dtype: float64
Billing KWH explained by PSI                             OLS Regression Results                            
Dep. Variable:            Billing KWH   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     150.2
Date:                Tue, 25 Aug 2026   Prob (F-statistic):           3.08e-34
Time:                       

In [8]:
# Standard Linear Regression
# PREDICTIVE
# Exploration of KWH and billing relationships
# Variables of interest: Predictors: Hours, Water Total, PSI, Metered DM, Power Factor, Billed DM, Fuel Rate
#                        Predicted: Billing KWH
predictors = ["Hours", "Water Total", "PSI", "Metered DM", "Power Factor", "Billed DM", "Fuel Rate"]
predicted = ["Billing KWH",]

# We need to split dataset based on time, rather than randomly. The last observation for the data is September 2025.
# Let's attempt to predict 6 months worth of data, so the training set will be start -> March 2025
X = main_df[predictors]
y = main_df[predicted]

print(main_df["Year-Month"])

train_X = main_df[main_df["Year-Month"] < "2025-04"]
train_X = train_X[predictors]

test_X = main_df[main_df["Year-Month"] >= "2025-04"]
test_X = test_X[predictors]

train_y = main_df[main_df["Year-Month"] < "2025-04"]
train_y = train_y[predicted]

test_y = main_df[main_df["Year-Month"] >= "2025-04"]
test_y = test_y[predicted]


# Normalization - NOTE: We are explaining rather than predicting in this case, so normalization should transform
#                       all data and not just the training data whose values are applied to the test data.
# NOTE: Normalization applies to PCA and other methods sensitive to absolute magnitude differences. 
#       OLS linear regression is not one of those.

0       2017-08
1       2017-09
2       2017-10
3       2017-11
4       2017-12
         ...   
8521    2025-05
8522    2025-06
8523    2025-07
8524    2025-08
8525    2025-09
Name: Year-Month, Length: 8526, dtype: object


In [ ]:
# Principle Components Analysis
# Principle Components Regression 